In [ ]:
import librosa
import numpy as np
import soundfile as sf

def preprocess_for_anc(file_path, target_sr=16000, latency_samples=50):
    """
    Prepares noise file for ANC train.

    Parameters:
    - file_path: Voice file path
    - target_sr: Target sample rate (16kHz for now can change later)
    - latency_samples: Compensation amount for hardware delay
    """

    # 1. Download and Resampling (Throw high frequency, relieve the processor)
    # We will decrease sample rate
    audio, sr = librosa.load(file_path, sr=target_sr)

    # 2. Normalization (Compress signal between -1 and +1)
    # It's important for stabile model learning.
    audio = audio / np.max(np.abs(audio))

    # 3. Create target for ANC.
    # For physical absorption: Noise + (-Noise) = 0
    target_signal = -1.0 * audio

    # 4. Delay Recovery (Predictive ANC)
    # There will be a delay (latency) until the model processes the sound and sends it to the speaker.
    # That's why we teach the model: “Look at the current sound and produce the inverse of what will happen 3ms later.”
    # We shift the target signal to the left (we are targeting the future).
    target_signal = np.roll(target_signal, -latency_samples)

    # Fill the gaps created after shifting with zeros.
    target_signal[-latency_samples:] = 0

    return audio, target_signal

# Use
input_noise, target_anti_noise = preprocess_for_anc("1-137-A-32.wav")

print(f"Input Noise Size: {input_noise.shape}")
print(f"Target (Anti-Noise) Signal Size: {target_anti_noise.shape}")